# Предсказания Qwen для тестовой выборки 🦋

Включите GPU и Internet в настройках Kaggle. Подключите датасет с `dataset.jsonl` и `submission-template.jsonl`, затем запустите ячейки по порядку. Итог: `/kaggle/working/submission.jsonl`.

## 1. Установка и настройки

In [ ]:
%pip -q install -U 'transformers>=4.57.0,<5' accelerate pillow requests tqdm

In [ ]:
from pathlib import Path
import json, re, math, os, hashlib, time
from io import BytesIO
from datetime import datetime, timezone
import requests
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

# Автопоиск в /kaggle/input; если файлов несколько, укажите точные пути.
DATASET_PATH = None     # пример: Path('/kaggle/input/ИМЯ/dataset.jsonl')
TEMPLATE_PATH = None    # пример: Path('/kaggle/input/ИМЯ/submission-template.jsonl')
MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
OUT = Path('/kaggle/working')
CHECKPOINT_PATH = OUT / 'qwen_test_checkpoint.jsonl'
LOG_PATH = OUT / 'qwen_test_errors.jsonl'
SUBMISSION_PATH = OUT / 'submission.jsonl'
MAX_NEW_TOKENS = 512
ATTEMPTS_PER_RUN = 3
QWEN_PROMPT = 'Find all adult butterflies and moths in the image.\n\nReturn one separate bounding box for each individual butterfly or moth,\neven when two insects overlap, touch, or are mating.\nDo not draw one combined box around multiple insects.\n\nEach box should tightly enclose the visible parts of one insect,\nincluding its wings, body, antennae, and legs when visible.\nIf an insect is partly hidden or cut off by the image edge,\nbox only the visible parts.\n\nDo not mark leaves, flowers, petals, shadows, bees, flies, beetles,\ncaterpillars, or other non-target objects.\nOnly return a box when you can identify an adult butterfly or moth.\nIf there are no butterflies or moths, return [].\n\nReturn only a JSON array, one object per insect:\n[{"bbox_2d": [x_min, y_min, x_max, y_max], "label": "butterfly"}]\n\nUse integer coordinates from 0 to 1000 relative to the full image.'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не обнаружен')
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU в настройках Kaggle и перезапустите сессию.')

## 2. Проверка входных данных

In [ ]:
def locate(path, name):
    if path is not None:
        result = Path(path)
        if not result.is_file(): raise FileNotFoundError(f'Не найден файл: {result}')
        return result
    matches = sorted(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Файл {name}: найдено {len(matches)}. Укажите путь вручную: {matches}')
    return matches[0]

def read_jsonl(path):
    rows = []
    with open(path, encoding='utf-8-sig') as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip(): continue
            try: item = json.loads(line)
            except json.JSONDecodeError as e: raise ValueError(f'{path}:{line_no}: {e}') from e
            if not isinstance(item, dict): raise ValueError(f'{path}:{line_no}: ожидается объект')
            rows.append(item)
    return rows

def validate_source(rows, label):
    seen = set()
    for r in rows:
        if 'photo_id' not in r or 'photo_url' not in r or not isinstance(r['photo_url'], str) or not r['photo_url'].startswith(('https://','http://')):
            raise ValueError(f'{label}: нужна запись с photo_id и ссылкой photo_url: {r}')
        key = str(r['photo_id'])
        if key in seen: raise ValueError(f'{label}: повтор photo_id {key}')
        seen.add(key)
    return seen

DATASET_PATH = locate(DATASET_PATH, 'dataset.jsonl')
TEMPLATE_PATH = locate(TEMPLATE_PATH, 'submission-template.jsonl')
dataset = read_jsonl(DATASET_PATH)
template = read_jsonl(TEMPLATE_PATH)
dataset_ids = validate_source(dataset, 'dataset')
template_ids = validate_source(template, 'template')
if not dataset or dataset_ids != template_ids: raise ValueError('Состав photo_id в датасете и шаблоне не совпадает.')
by_id = {str(r['photo_id']): r for r in dataset}
for row in template:
    if set(row) != {'photo_id','photo_url','boxes','error'}: raise ValueError(f'Неожиданные поля шаблона: {list(row)}')
    if row['photo_url'] != by_id[str(row['photo_id'])]['photo_url']: raise ValueError(f"Разные ссылки: {row['photo_id']}")

# Сохраняем порядок и тип photo_id из шаблона.
ordered = [{**r, 'photo_url': by_id[str(r['photo_id'])]['photo_url']} for r in template]
run_config = {'model':MODEL_ID, 'prompt':QWEN_PROMPT, 'max_tokens':MAX_NEW_TOKENS,
              'dataset':[(str(r['photo_id']),r['photo_url']) for r in ordered]}
run_key = hashlib.sha256(json.dumps(run_config,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
print('Датасет:', DATASET_PATH, '| Шаблон:', TEMPLATE_PATH)
print('Фотографий:',len(ordered), '| Порядок как в шаблоне')

## 3. Загрузка модели

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map='auto'
).eval()
print('Модель загружена:', MODEL_ID)

## 4. Предсказания
Прогресс сохраняется после каждого изображения. Повторный запуск этой ячейки обрабатывает только пропущенные и неудачные фотографии.

In [ ]:
ALLOWED_LABELS = {'butterfly','butterflies','adult butterfly','adult butterflies',
                  'moth','moths','adult moth','adult moths','adult butterfly and moth'}
session = requests.Session()

def predict(image):
    messages = [{'role':'user','content':[{'type':'image','image':image},
                                          {'type':'text','text':QWEN_PROMPT}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                          return_dict=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    tokens = output[0][inputs['input_ids'].shape[-1]:]
    raw = processor.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    if len(tokens) >= MAX_NEW_TOKENS and tokens[-1].item() != processor.tokenizer.eos_token_id:
        raise ValueError('Ответ модели достиг лимита токенов и, возможно, обрезан: '+raw[:1000])
    clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.IGNORECASE).strip()
    try: items = json.loads(clean)
    except json.JSONDecodeError as exc: raise ValueError(f'Ответ не является JSON: {exc}; ответ: {raw[:1000]}') from exc
    if not isinstance(items,list): raise ValueError(f'Ожидался JSON-массив: {raw[:1000]}')
    result=[]
    for item in items:
        if not isinstance(item,dict) or not isinstance(item.get('bbox_2d'),list) or len(item['bbox_2d'])!=4:
            raise ValueError(f'Некорректный bbox_2d: {item}')
        label=str(item.get('label','butterfly')).strip().lower()
        if label not in ALLOWED_LABELS: raise ValueError(f'Неизвестная метка: {label!r}')
        x1,y1,x2,y2=map(float,item['bbox_2d'])
        if not all(math.isfinite(x) and 0<=x<=1000 for x in (x1,y1,x2,y2)) or x1>=x2 or y1>=y2:
            raise ValueError(f'Некорректные координаты bbox_2d: {item["bbox_2d"]}')
        result.append({'left':x1/1000, 'top':y1/1000,
                       'width':(x2-x1)/1000, 'height':(y2-y1)/1000})
    return result

def log_error(row, attempt, exc):
    obj={'photo_id':row['photo_id'],'photo_url':row['photo_url'],'attempt':attempt,
         'error':f'{type(exc).__name__}: {exc}', 'time':datetime.now(timezone.utc).isoformat()}
    with LOG_PATH.open('a',encoding='utf-8') as f: f.write(json.dumps(obj,ensure_ascii=False)+'\n')

completed={}
if CHECKPOINT_PATH.exists():
    for entry in read_jsonl(CHECKPOINT_PATH):
        if entry.get('run_key') != run_key:
            raise ValueError('Checkpoint относится к другому набору или промпту. Переименуйте файл прогресса перед новым запуском.')
        pred=entry.get('result',{}); key=str(pred.get('photo_id'))
        if key not in dataset_ids or pred.get('photo_url') != by_id[key]['photo_url'] or not isinstance(pred.get('boxes'),list) or pred.get('error') is not None:
            raise ValueError(f'Некорректная запись checkpoint: {key}')
        completed[key]=pred
print('Уже обработано:',len(completed),'из',len(ordered))
for row in tqdm(ordered,desc='Qwen',unit='фото'):
    key=str(row['photo_id'])
    if key in completed: continue
    for attempt in range(1,ATTEMPTS_PER_RUN+1):
        try:
            response=session.get(row['photo_url'],timeout=(15,60))
            response.raise_for_status()
            with Image.open(BytesIO(response.content)) as im:
                image=ImageOps.exif_transpose(im).convert('RGB')
            boxes=predict(image)
            result={'photo_id':row['photo_id'],'photo_url':row['photo_url'],'boxes':boxes,'error':None}
            with CHECKPOINT_PATH.open('a',encoding='utf-8') as f:
                f.write(json.dumps({'run_key':run_key,'result':result},ensure_ascii=False,allow_nan=False)+'\n')
                f.flush()
            completed[key]=result
            break
        except Exception as exc:
            log_error(row,attempt,exc)
            if attempt < ATTEMPTS_PER_RUN: time.sleep(min(2**attempt,8))
            else: print(f'\nНе удалось обработать {key}: {type(exc).__name__}: {str(exc)[:200]}')
print(f'Готово: {len(completed)}/{len(ordered)}; осталось: {len(ordered)-len(completed)}')
print('Прогресс:',CHECKPOINT_PATH,'| Журнал ошибок:',LOG_PATH)

## 5. Проверка и сохранение submission
Если остались ошибки, повторите предыдущую ячейку. Ошибка не заменяется пустым списком.

In [ ]:
missing = [str(row['photo_id']) for row in ordered if str(row['photo_id']) not in completed]
if missing:
    raise RuntimeError(f'Готовый submission не создан: не обработано {len(missing)} фото. Примеры: {missing[:10]}. Повторите предыдущую ячейку; причины — в журнале ошибок.')
results=[completed[str(row['photo_id'])] for row in ordered]
for row in results:
    if set(row) != {'photo_id','photo_url','boxes','error'} or row['error'] is not None or not isinstance(row['boxes'],list):
        raise ValueError(f'Некорректная запись: {row}')
    for b in row['boxes']:
        if set(b) != {'left','top','width','height'}: raise ValueError(f'Некорректные поля рамки: {b}')
        x,y,w,h=(float(b[k]) for k in ('left','top','width','height'))
        if not all(math.isfinite(v) for v in (x,y,w,h)) or not (0<=x<1 and 0<=y<1 and w>0 and h>0 and x+w<=1.0000001 and y+h<=1.0000001):
            raise ValueError(f'Рамка вне диапазона: {b}')
if len(results)!=len(template) or len({str(r['photo_id']) for r in results})!=len(template): raise ValueError('Несовпадение количества или повтор photo_id')
# Записываем во временный файл и заменяем только полностью готовым результатом.
tmp=SUBMISSION_PATH.with_suffix('.jsonl.tmp')
with tmp.open('w',encoding='utf-8') as f:
    for row in results: f.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n')
if tmp.stat().st_size > 5*1024*1024: raise ValueError('Файл превышает 5 МиБ')
os.replace(tmp,SUBMISSION_PATH)
print('Сохранено:',SUBMISSION_PATH)
print('Строк:',len(results),'| Рамок:',sum(len(r['boxes']) for r in results),
      '| Фото без бабочек:',sum(not r['boxes'] for r in results))
print('Первые две строки:')
for row in results[:2]: print(json.dumps(row,ensure_ascii=False))
print('Скачайте submission.jsonl в панели Output / Files справа.')